In [0]:
%pip install holidays --quiet
dbutils.library.restartPython()

# Supply Chain Demo Data for Supervisor Agent

This notebook creates synthetic supply chain data across **two domains**, each backing a separate Genie space:

| Domain | Genie Space | Tables | Metric View |
|--------|------------|--------|-----------|
| Procurement & Inventory | Space 1 | suppliers, materials, purchase_orders, inventory | procurement_metrics |
| Logistics & Fulfillment | Space 2 | carriers, shipments, routes, delivery_events | logistics_metrics |

A **supervisor agent** (notebook 03) routes user questions to the correct Genie space based on intent.

### Metric View Concepts
| Concept | What it does | Example |
|---------|-------------|---------|
| **Measure** | KPI — aggregates across rows | `SUM(quantity * unit_cost)` → Total Spend |
| **Dimension** | Attribute for grouping/slicing | `DATE_TRUNC('MONTH', order_date)` → Order Month |
| **Filter** | Condition to narrow rows | `status != 'Cancelled'` |

In [0]:
from datetime import date, timedelta, datetime
import random
import holidays
from pyspark.sql import functions as F

random.seed(42)

CATALOG = "ram"
SCHEMA = "supply_chain_demo"

# Table names - Domain 1: Procurement & Inventory
SUPPLIERS_TABLE = f"{CATALOG}.{SCHEMA}.suppliers"
MATERIALS_TABLE = f"{CATALOG}.{SCHEMA}.materials"
PURCHASE_ORDERS_TABLE = f"{CATALOG}.{SCHEMA}.purchase_orders"
INVENTORY_TABLE = f"{CATALOG}.{SCHEMA}.inventory"

# Table names - Domain 2: Logistics & Fulfillment
CARRIERS_TABLE = f"{CATALOG}.{SCHEMA}.carriers"
SHIPMENTS_TABLE = f"{CATALOG}.{SCHEMA}.shipments"
ROUTES_TABLE = f"{CATALOG}.{SCHEMA}.routes"
DELIVERY_EVENTS_TABLE = f"{CATALOG}.{SCHEMA}.delivery_events"

# Metric views
PROCUREMENT_METRICS = f"{CATALOG}.{SCHEMA}.procurement_metrics"
LOGISTICS_METRICS = f"{CATALOG}.{SCHEMA}.logistics_metrics"

# Time window
today = date.today()
six_months_ago = today - timedelta(days=180)
four_months_ago = today - timedelta(days=120)

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Order window: {six_months_ago} to {today}")
print(f"Shipment window: {four_months_ago} to {today}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✓ Schema ready: {CATALOG}.{SCHEMA}")

---
## Domain 1: Procurement & Inventory
Tables for tracking suppliers, raw materials, purchase orders, and warehouse inventory levels.

In [0]:
suppliers_data = [
    {"supplier_id": "S001", "supplier_name": "Apex Steel Corp", "country": "US", "lead_time_days": 7, "reliability_score": 94, "category": "Raw Materials"},
    {"supplier_id": "S002", "supplier_name": "Shanghai Components Ltd", "country": "China", "lead_time_days": 21, "reliability_score": 87, "category": "Components"},
    {"supplier_id": "S003", "supplier_name": "Bayern Precision GmbH", "country": "Germany", "lead_time_days": 14, "reliability_score": 96, "category": "Components"},
    {"supplier_id": "S004", "supplier_name": "Monterrey Packaging SA", "country": "Mexico", "lead_time_days": 5, "reliability_score": 91, "category": "Packaging"},
    {"supplier_id": "S005", "supplier_name": "Osaka Electronics Co", "country": "Japan", "lead_time_days": 18, "reliability_score": 98, "category": "Components"},
    {"supplier_id": "S006", "supplier_name": "Great Lakes Metals", "country": "US", "lead_time_days": 6, "reliability_score": 89, "category": "Raw Materials"},
    {"supplier_id": "S007", "supplier_name": "Shenzhen Circuit Works", "country": "China", "lead_time_days": 25, "reliability_score": 82, "category": "Components"},
    {"supplier_id": "S008", "supplier_name": "EcoPack Solutions", "country": "US", "lead_time_days": 4, "reliability_score": 93, "category": "Packaging"},
]

df = spark.createDataFrame(suppliers_data)
df.write.mode("overwrite").saveAsTable(SUPPLIERS_TABLE)
print(f"✓ {SUPPLIERS_TABLE}: {df.count()} rows")

In [0]:
materials_data = [
    {"material_id": "M001", "material_name": "Cold Rolled Steel", "category": "Raw Material", "unit_of_measure": "kg", "standard_cost": 2.45},
    {"material_id": "M002", "material_name": "Aluminum Sheet 6061", "category": "Raw Material", "unit_of_measure": "kg", "standard_cost": 4.80},
    {"material_id": "M003", "material_name": "Circuit Board Assembly", "category": "Component", "unit_of_measure": "unit", "standard_cost": 18.50},
    {"material_id": "M004", "material_name": "Servo Motor Unit", "category": "Component", "unit_of_measure": "unit", "standard_cost": 45.00},
    {"material_id": "M005", "material_name": "Corrugated Cardboard", "category": "Packaging", "unit_of_measure": "sheet", "standard_cost": 0.85},
    {"material_id": "M006", "material_name": "Copper Wire Harness", "category": "Component", "unit_of_measure": "unit", "standard_cost": 12.30},
    {"material_id": "M007", "material_name": "Stainless Steel Fasteners", "category": "Raw Material", "unit_of_measure": "kg", "standard_cost": 6.10},
    {"material_id": "M008", "material_name": "LCD Display Panel", "category": "Component", "unit_of_measure": "unit", "standard_cost": 32.00},
    {"material_id": "M009", "material_name": "Foam Insert Padding", "category": "Packaging", "unit_of_measure": "sheet", "standard_cost": 1.20},
    {"material_id": "M010", "material_name": "Lithium Battery Cell", "category": "Component", "unit_of_measure": "unit", "standard_cost": 8.75},
    {"material_id": "M011", "material_name": "Industrial Adhesive", "category": "Raw Material", "unit_of_measure": "liter", "standard_cost": 15.40},
    {"material_id": "M012", "material_name": "Shrink Wrap Film", "category": "Packaging", "unit_of_measure": "roll", "standard_cost": 3.25},
]

df = spark.createDataFrame(materials_data)
df.write.mode("overwrite").saveAsTable(MATERIALS_TABLE)
print(f"✓ {MATERIALS_TABLE}: {df.count()} rows")

In [0]:
# Generate 30 purchase orders over the last 6 months
statuses = ["Delivered", "Delivered", "Delivered", "Delivered", "In Transit", "In Transit", "Pending", "Cancelled"]
supplier_ids = [s["supplier_id"] for s in suppliers_data]
material_ids = [m["material_id"] for m in materials_data]
supplier_materials = {
    "S001": ["M001", "M007"], "S002": ["M003", "M006", "M010"],
    "S003": ["M004", "M008"], "S004": ["M005", "M009", "M012"],
    "S005": ["M003", "M008", "M010"], "S006": ["M001", "M002", "M007"],
    "S007": ["M006", "M010"], "S008": ["M005", "M009", "M012"],
}
supplier_lead = {s["supplier_id"]: s["lead_time_days"] for s in suppliers_data}

purchase_orders_data = []
for i in range(1, 31):
    po_id = f"PO{i:03d}"
    supplier_id = random.choice(supplier_ids)
    material_id = random.choice(supplier_materials[supplier_id])
    quantity = random.choice([100, 200, 250, 500, 750, 1000, 1500, 2000])
    
    # Cost varies slightly from standard
    std_cost = next(m["standard_cost"] for m in materials_data if m["material_id"] == material_id)
    unit_cost = round(std_cost * random.uniform(0.90, 1.10), 2)
    
    days_back = random.randint(10, 180)
    order_date = today - timedelta(days=days_back)
    lead = supplier_lead[supplier_id]
    expected_delivery = order_date + timedelta(days=lead)
    
    status = random.choice(statuses)
    if status == "Delivered":
        delay = random.randint(-2, 5)
        actual_delivery = str(expected_delivery + timedelta(days=delay))
    elif status == "In Transit":
        actual_delivery = None
    elif status == "Pending":
        actual_delivery = None
    else:  # Cancelled
        actual_delivery = None
    
    purchase_orders_data.append({
        "po_id": po_id, "supplier_id": supplier_id, "material_id": material_id,
        "quantity": quantity, "unit_cost": unit_cost,
        "order_date": str(order_date), "expected_delivery": str(expected_delivery),
        "actual_delivery": actual_delivery, "status": status
    })

df = spark.createDataFrame(purchase_orders_data)
df = df.withColumn("order_date", F.col("order_date").cast("date")) \
       .withColumn("expected_delivery", F.col("expected_delivery").cast("date")) \
       .withColumn("actual_delivery", F.col("actual_delivery").cast("date"))
df.write.mode("overwrite").saveAsTable(PURCHASE_ORDERS_TABLE)
print(f"✓ {PURCHASE_ORDERS_TABLE}: {df.count()} rows")

In [0]:
warehouses = ["WH-EAST", "WH-WEST", "WH-CENTRAL"]

inventory_data = []
inv_id = 1
for wh in warehouses:
    for mat in materials_data:
        qty_on_hand = random.randint(50, 5000)
        reorder_point = random.randint(100, 500)
        safety_stock = int(reorder_point * 0.5)
        days_since_replenish = random.randint(1, 30)
        last_replenished = str(today - timedelta(days=days_since_replenish))
        
        inventory_data.append({
            "inventory_id": f"INV{inv_id:04d}",
            "warehouse_id": wh,
            "material_id": mat["material_id"],
            "quantity_on_hand": qty_on_hand,
            "reorder_point": reorder_point,
            "safety_stock": safety_stock,
            "last_replenished": last_replenished
        })
        inv_id += 1

df = spark.createDataFrame(inventory_data)
df = df.withColumn("last_replenished", F.col("last_replenished").cast("date"))
df.write.mode("overwrite").saveAsTable(INVENTORY_TABLE)
print(f"✓ {INVENTORY_TABLE}: {df.count()} rows")

---
## Domain 2: Logistics & Fulfillment
Tables for tracking carriers, shipments, routes, and delivery milestones.

In [0]:
carriers_data = [
    {"carrier_id": "CR001", "carrier_name": "FastFreight Logistics", "transport_mode": "Truck", "cost_per_kg": 0.45, "on_time_delivery_pct": 92.5, "service_region": "Domestic"},
    {"carrier_id": "CR002", "carrier_name": "Pacific Ocean Lines", "transport_mode": "Ocean", "cost_per_kg": 0.12, "on_time_delivery_pct": 78.0, "service_region": "International"},
    {"carrier_id": "CR003", "carrier_name": "SkyBridge Air Cargo", "transport_mode": "Air", "cost_per_kg": 2.80, "on_time_delivery_pct": 96.0, "service_region": "Both"},
    {"carrier_id": "CR004", "carrier_name": "Continental Rail Express", "transport_mode": "Rail", "cost_per_kg": 0.22, "on_time_delivery_pct": 85.0, "service_region": "Domestic"},
    {"carrier_id": "CR005", "carrier_name": "Global Express Shipping", "transport_mode": "Truck", "cost_per_kg": 0.55, "on_time_delivery_pct": 94.0, "service_region": "Both"},
    {"carrier_id": "CR006", "carrier_name": "TransAsia Freight", "transport_mode": "Ocean", "cost_per_kg": 0.10, "on_time_delivery_pct": 74.0, "service_region": "International"},
]

df = spark.createDataFrame(carriers_data)
df.write.mode("overwrite").saveAsTable(CARRIERS_TABLE)
print(f"✓ {CARRIERS_TABLE}: {df.count()} rows")

In [0]:
destinations = [
    "Chicago, IL", "Los Angeles, CA", "Houston, TX", "Atlanta, GA",
    "Seattle, WA", "Denver, CO", "Miami, FL", "Detroit, MI",
    "Phoenix, AZ", "Portland, OR", "Shanghai, CN", "Munich, DE", "Tokyo, JP"
]
carrier_ids = [c["carrier_id"] for c in carriers_data]
ship_statuses = ["Delivered", "Delivered", "Delivered", "Delivered", "Delivered",
                 "In Transit", "In Transit", "Delayed", "Returned"]

shipments_data = []
for i in range(1, 41):
    shipment_id = f"SH{i:03d}"
    origin = random.choice(warehouses)
    dest = random.choice(destinations)
    carrier_id = random.choice(carrier_ids)
    
    days_back = random.randint(5, 120)
    ship_date = today - timedelta(days=days_back)
    transit_days = random.randint(2, 21)
    est_delivery = ship_date + timedelta(days=transit_days)
    
    status = random.choice(ship_statuses)
    if status == "Delivered":
        delay = random.randint(-1, 3)
        actual_delivery = str(est_delivery + timedelta(days=delay))
    elif status == "Delayed":
        actual_delivery = str(est_delivery + timedelta(days=random.randint(4, 10)))
    else:
        actual_delivery = None
    
    weight_kg = round(random.uniform(50, 5000), 1)
    cost_per_kg = next(c["cost_per_kg"] for c in carriers_data if c["carrier_id"] == carrier_id)
    shipping_cost = round(weight_kg * cost_per_kg * random.uniform(0.9, 1.15), 2)
    
    shipments_data.append({
        "shipment_id": shipment_id, "origin_warehouse": origin,
        "destination_city": dest, "carrier_id": carrier_id,
        "ship_date": str(ship_date), "estimated_delivery": str(est_delivery),
        "actual_delivery": actual_delivery, "status": status,
        "weight_kg": weight_kg, "shipping_cost": shipping_cost
    })

df = spark.createDataFrame(shipments_data)
df = df.withColumn("ship_date", F.col("ship_date").cast("date")) \
       .withColumn("estimated_delivery", F.col("estimated_delivery").cast("date")) \
       .withColumn("actual_delivery", F.col("actual_delivery").cast("date"))
df.write.mode("overwrite").saveAsTable(SHIPMENTS_TABLE)
print(f"✓ {SHIPMENTS_TABLE}: {df.count()} rows")

In [0]:
routes_data = [
    {"route_id": "RT001", "origin": "WH-EAST", "destination": "Chicago, IL", "distance_km": 1200, "avg_transit_days": 3, "transport_mode": "Truck"},
    {"route_id": "RT002", "origin": "WH-EAST", "destination": "Atlanta, GA", "distance_km": 900, "avg_transit_days": 2, "transport_mode": "Truck"},
    {"route_id": "RT003", "origin": "WH-WEST", "destination": "Los Angeles, CA", "distance_km": 600, "avg_transit_days": 1, "transport_mode": "Truck"},
    {"route_id": "RT004", "origin": "WH-WEST", "destination": "Seattle, WA", "distance_km": 1100, "avg_transit_days": 2, "transport_mode": "Truck"},
    {"route_id": "RT005", "origin": "WH-CENTRAL", "destination": "Houston, TX", "distance_km": 800, "avg_transit_days": 2, "transport_mode": "Truck"},
    {"route_id": "RT006", "origin": "WH-CENTRAL", "destination": "Denver, CO", "distance_km": 950, "avg_transit_days": 2, "transport_mode": "Rail"},
    {"route_id": "RT007", "origin": "WH-EAST", "destination": "Miami, FL", "distance_km": 1800, "avg_transit_days": 4, "transport_mode": "Rail"},
    {"route_id": "RT008", "origin": "WH-WEST", "destination": "Shanghai, CN", "distance_km": 9500, "avg_transit_days": 18, "transport_mode": "Ocean"},
    {"route_id": "RT009", "origin": "WH-EAST", "destination": "Munich, DE", "distance_km": 7200, "avg_transit_days": 14, "transport_mode": "Ocean"},
    {"route_id": "RT010", "origin": "WH-CENTRAL", "destination": "Tokyo, JP", "distance_km": 10200, "avg_transit_days": 5, "transport_mode": "Air"},
    {"route_id": "RT011", "origin": "WH-EAST", "destination": "Detroit, MI", "distance_km": 750, "avg_transit_days": 2, "transport_mode": "Truck"},
    {"route_id": "RT012", "origin": "WH-WEST", "destination": "Phoenix, AZ", "distance_km": 500, "avg_transit_days": 1, "transport_mode": "Truck"},
]

df = spark.createDataFrame(routes_data)
df.write.mode("overwrite").saveAsTable(ROUTES_TABLE)
print(f"✓ {ROUTES_TABLE}: {df.count()} rows")

In [0]:
event_types = ["Picked Up", "In Transit", "Out for Delivery", "Delivered", "Delay Reported", "Returned"]
locations = ["Origin Warehouse", "Regional Hub", "Distribution Center", "Local Depot", "Customer Location", "Customs Hold", "Port Terminal"]

delivery_events_data = []
event_id = 1

for ship in shipments_data:
    ship_dt = datetime.strptime(ship["ship_date"], "%Y-%m-%d")
    
    # Picked up event
    delivery_events_data.append({
        "event_id": f"EV{event_id:05d}", "shipment_id": ship["shipment_id"],
        "event_timestamp": str(ship_dt + timedelta(hours=random.randint(2, 8))),
        "event_type": "Picked Up", "location": "Origin Warehouse", "notes": None
    })
    event_id += 1
    
    # In Transit event
    delivery_events_data.append({
        "event_id": f"EV{event_id:05d}", "shipment_id": ship["shipment_id"],
        "event_timestamp": str(ship_dt + timedelta(days=1, hours=random.randint(4, 12))),
        "event_type": "In Transit", "location": random.choice(["Regional Hub", "Distribution Center", "Port Terminal"]),
        "notes": None
    })
    event_id += 1
    
    if ship["status"] == "Delivered":
        # Out for Delivery
        delivery_dt = datetime.strptime(ship["actual_delivery"], "%Y-%m-%d")
        delivery_events_data.append({
            "event_id": f"EV{event_id:05d}", "shipment_id": ship["shipment_id"],
            "event_timestamp": str(delivery_dt - timedelta(hours=random.randint(3, 6))),
            "event_type": "Out for Delivery", "location": "Local Depot", "notes": None
        })
        event_id += 1
        # Delivered
        delivery_events_data.append({
            "event_id": f"EV{event_id:05d}", "shipment_id": ship["shipment_id"],
            "event_timestamp": str(delivery_dt + timedelta(hours=random.randint(9, 17))),
            "event_type": "Delivered", "location": "Customer Location",
            "notes": random.choice([None, "Signed by recipient", "Left at dock", "Verified quantity"])
        })
        event_id += 1
    elif ship["status"] == "Delayed":
        delivery_events_data.append({
            "event_id": f"EV{event_id:05d}", "shipment_id": ship["shipment_id"],
            "event_timestamp": str(ship_dt + timedelta(days=random.randint(2, 5), hours=10)),
            "event_type": "Delay Reported", "location": random.choice(["Customs Hold", "Regional Hub", "Port Terminal"]),
            "notes": random.choice(["Weather delay", "Customs clearance pending", "Carrier capacity issue", "Port congestion"])
        })
        event_id += 1
        # Still delivered eventually
        delivery_dt = datetime.strptime(ship["actual_delivery"], "%Y-%m-%d")
        delivery_events_data.append({
            "event_id": f"EV{event_id:05d}", "shipment_id": ship["shipment_id"],
            "event_timestamp": str(delivery_dt + timedelta(hours=14)),
            "event_type": "Delivered", "location": "Customer Location", "notes": "Delivered after delay"
        })
        event_id += 1
    elif ship["status"] == "Returned":
        delivery_events_data.append({
            "event_id": f"EV{event_id:05d}", "shipment_id": ship["shipment_id"],
            "event_timestamp": str(ship_dt + timedelta(days=random.randint(3, 7), hours=11)),
            "event_type": "Returned", "location": "Distribution Center",
            "notes": random.choice(["Damaged in transit", "Refused by customer", "Wrong address"])
        })
        event_id += 1

df = spark.createDataFrame(delivery_events_data)
df = df.withColumn("event_timestamp", F.col("event_timestamp").cast("timestamp"))
df.write.mode("overwrite").saveAsTable(DELIVERY_EVENTS_TABLE)
print(f"✓ {DELIVERY_EVENTS_TABLE}: {df.count()} rows")

---
## Table & Column Comments
Comments help Genie understand the data. Every table and column gets a description.

In [0]:
# ============================================================
# DOMAIN 1: Procurement & Inventory comments
# ============================================================

# Suppliers
spark.sql(f"COMMENT ON TABLE {SUPPLIERS_TABLE} IS 'Supplier master data. Contains vendor information, country of origin, lead times, and reliability scores. Primary key: supplier_id.'")
spark.sql(f"ALTER TABLE {SUPPLIERS_TABLE} ALTER COLUMN supplier_id COMMENT 'Unique supplier identifier (S001-S008)'")
spark.sql(f"ALTER TABLE {SUPPLIERS_TABLE} ALTER COLUMN supplier_name COMMENT 'Legal name of the supplier company'")
spark.sql(f"ALTER TABLE {SUPPLIERS_TABLE} ALTER COLUMN country COMMENT 'Country where supplier is headquartered (US, China, Germany, Mexico, Japan)'")
spark.sql(f"ALTER TABLE {SUPPLIERS_TABLE} ALTER COLUMN lead_time_days COMMENT 'Average number of days from order to delivery'")
spark.sql(f"ALTER TABLE {SUPPLIERS_TABLE} ALTER COLUMN reliability_score COMMENT 'Supplier reliability rating 0-100 (higher is better)'")
spark.sql(f"ALTER TABLE {SUPPLIERS_TABLE} ALTER COLUMN category COMMENT 'Supplier specialization: Raw Materials, Components, or Packaging'")

# Materials
spark.sql(f"COMMENT ON TABLE {MATERIALS_TABLE} IS 'Bill of materials catalog. Lists all raw materials, components, and packaging used in manufacturing. Primary key: material_id.'")
spark.sql(f"ALTER TABLE {MATERIALS_TABLE} ALTER COLUMN material_id COMMENT 'Unique material identifier (M001-M012)'")
spark.sql(f"ALTER TABLE {MATERIALS_TABLE} ALTER COLUMN material_name COMMENT 'Descriptive name of the material'")
spark.sql(f"ALTER TABLE {MATERIALS_TABLE} ALTER COLUMN category COMMENT 'Material type: Raw Material, Component, or Packaging'")
spark.sql(f"ALTER TABLE {MATERIALS_TABLE} ALTER COLUMN unit_of_measure COMMENT 'Unit for ordering/stocking: kg, unit, sheet, roll, liter'")
spark.sql(f"ALTER TABLE {MATERIALS_TABLE} ALTER COLUMN standard_cost COMMENT 'Standard unit cost in USD for budgeting purposes'")

# Purchase Orders
spark.sql(f"COMMENT ON TABLE {PURCHASE_ORDERS_TABLE} IS 'Purchase order transactions with suppliers. Tracks quantities, costs, delivery dates, and fulfillment status. Primary key: po_id.'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN po_id COMMENT 'Unique purchase order number (PO001-PO030)'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN supplier_id COMMENT 'Foreign key to suppliers table'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN material_id COMMENT 'Foreign key to materials table'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN quantity COMMENT 'Number of units ordered'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN unit_cost COMMENT 'Actual negotiated cost per unit in USD'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN order_date COMMENT 'Date the purchase order was placed'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN expected_delivery COMMENT 'Estimated delivery date based on supplier lead time'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN actual_delivery COMMENT 'Actual delivery date. NULL if order is still open (In Transit, Pending, or Cancelled)'")
spark.sql(f"ALTER TABLE {PURCHASE_ORDERS_TABLE} ALTER COLUMN status COMMENT 'Order status: Delivered, In Transit, Pending, or Cancelled'")

# Inventory
spark.sql(f"COMMENT ON TABLE {INVENTORY_TABLE} IS 'Current warehouse inventory levels by material. Tracks stock quantities against reorder points and safety stock. Primary key: inventory_id.'")
spark.sql(f"ALTER TABLE {INVENTORY_TABLE} ALTER COLUMN inventory_id COMMENT 'Unique inventory record identifier'")
spark.sql(f"ALTER TABLE {INVENTORY_TABLE} ALTER COLUMN warehouse_id COMMENT 'Warehouse location: WH-EAST, WH-WEST, or WH-CENTRAL'")
spark.sql(f"ALTER TABLE {INVENTORY_TABLE} ALTER COLUMN material_id COMMENT 'Foreign key to materials table'")
spark.sql(f"ALTER TABLE {INVENTORY_TABLE} ALTER COLUMN quantity_on_hand COMMENT 'Current available stock in warehouse'")
spark.sql(f"ALTER TABLE {INVENTORY_TABLE} ALTER COLUMN reorder_point COMMENT 'Stock level that triggers a new purchase order'")
spark.sql(f"ALTER TABLE {INVENTORY_TABLE} ALTER COLUMN safety_stock COMMENT 'Minimum stock level to prevent stockouts (50% of reorder point)'")
spark.sql(f"ALTER TABLE {INVENTORY_TABLE} ALTER COLUMN last_replenished COMMENT 'Date when stock was last replenished'")

print("✓ Domain 1 comments applied (suppliers, materials, purchase_orders, inventory)")

# ============================================================
# DOMAIN 2: Logistics & Fulfillment comments
# ============================================================

# Carriers
spark.sql(f"COMMENT ON TABLE {CARRIERS_TABLE} IS 'Carrier/freight provider master data. Contains transport modes, pricing, and delivery performance. Primary key: carrier_id.'")
spark.sql(f"ALTER TABLE {CARRIERS_TABLE} ALTER COLUMN carrier_id COMMENT 'Unique carrier identifier (CR001-CR006)'")
spark.sql(f"ALTER TABLE {CARRIERS_TABLE} ALTER COLUMN carrier_name COMMENT 'Carrier company name'")
spark.sql(f"ALTER TABLE {CARRIERS_TABLE} ALTER COLUMN transport_mode COMMENT 'Primary mode of transport: Truck, Rail, Air, or Ocean'")
spark.sql(f"ALTER TABLE {CARRIERS_TABLE} ALTER COLUMN cost_per_kg COMMENT 'Base shipping rate in USD per kilogram'")
spark.sql(f"ALTER TABLE {CARRIERS_TABLE} ALTER COLUMN on_time_delivery_pct COMMENT 'Historical on-time delivery percentage (0-100)'")
spark.sql(f"ALTER TABLE {CARRIERS_TABLE} ALTER COLUMN service_region COMMENT 'Coverage area: Domestic, International, or Both'")

# Shipments
spark.sql(f"COMMENT ON TABLE {SHIPMENTS_TABLE} IS 'Outbound shipment records from warehouses to customers/destinations. Tracks carrier, dates, weight, cost, and delivery status. Primary key: shipment_id.'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN shipment_id COMMENT 'Unique shipment tracking number (SH001-SH040)'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN origin_warehouse COMMENT 'Warehouse shipment originated from: WH-EAST, WH-WEST, or WH-CENTRAL'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN destination_city COMMENT 'Delivery destination city and state/country'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN carrier_id COMMENT 'Foreign key to carriers table'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN ship_date COMMENT 'Date shipment departed the warehouse'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN estimated_delivery COMMENT 'Expected delivery date at destination'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN actual_delivery COMMENT 'Actual delivery date. NULL if shipment is still in transit'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN status COMMENT 'Shipment status: Delivered, In Transit, Delayed, or Returned'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN weight_kg COMMENT 'Total shipment weight in kilograms'")
spark.sql(f"ALTER TABLE {SHIPMENTS_TABLE} ALTER COLUMN shipping_cost COMMENT 'Total shipping cost in USD'")

# Routes
spark.sql(f"COMMENT ON TABLE {ROUTES_TABLE} IS 'Defined shipping routes between warehouses and destinations. Contains distance, transit time, and transport mode. Primary key: route_id.'")
spark.sql(f"ALTER TABLE {ROUTES_TABLE} ALTER COLUMN route_id COMMENT 'Unique route identifier (RT001-RT012)'")
spark.sql(f"ALTER TABLE {ROUTES_TABLE} ALTER COLUMN origin COMMENT 'Starting warehouse for the route'")
spark.sql(f"ALTER TABLE {ROUTES_TABLE} ALTER COLUMN destination COMMENT 'End destination city'")
spark.sql(f"ALTER TABLE {ROUTES_TABLE} ALTER COLUMN distance_km COMMENT 'Total route distance in kilometers'")
spark.sql(f"ALTER TABLE {ROUTES_TABLE} ALTER COLUMN avg_transit_days COMMENT 'Average transit time in days for this route'")
spark.sql(f"ALTER TABLE {ROUTES_TABLE} ALTER COLUMN transport_mode COMMENT 'Transport mode used on this route: Truck, Rail, Air, or Ocean'")

# Delivery Events
spark.sql(f"COMMENT ON TABLE {DELIVERY_EVENTS_TABLE} IS 'Shipment milestone/tracking events. Each row is one status update for a shipment. Primary key: event_id.'")
spark.sql(f"ALTER TABLE {DELIVERY_EVENTS_TABLE} ALTER COLUMN event_id COMMENT 'Unique event identifier'")
spark.sql(f"ALTER TABLE {DELIVERY_EVENTS_TABLE} ALTER COLUMN shipment_id COMMENT 'Foreign key to shipments table'")
spark.sql(f"ALTER TABLE {DELIVERY_EVENTS_TABLE} ALTER COLUMN event_timestamp COMMENT 'Timestamp when this event occurred'")
spark.sql(f"ALTER TABLE {DELIVERY_EVENTS_TABLE} ALTER COLUMN event_type COMMENT 'Event milestone: Picked Up, In Transit, Out for Delivery, Delivered, Delay Reported, or Returned'")
spark.sql(f"ALTER TABLE {DELIVERY_EVENTS_TABLE} ALTER COLUMN location COMMENT 'Location where event occurred (e.g., Regional Hub, Customer Location)'")
spark.sql(f"ALTER TABLE {DELIVERY_EVENTS_TABLE} ALTER COLUMN notes COMMENT 'Optional notes about the event (delay reason, delivery confirmation)'")

print("✓ Domain 2 comments applied (carriers, shipments, routes, delivery_events)")

---
## Metric Views (Semantic Layer)
Each domain gets a metric view so Genie natively understands the KPIs without needing separate knowledge snippets.

| Concept | Procurement Example | Logistics Example |
|---------|--------------------|---------|
| **Measure** | `SUM(quantity * unit_cost)` → Total Spend | `SUM(shipping_cost)` → Total Shipping Cost |
| **Dimension** | `supplier_name` | `carrier_name` |
| **Filter** | `status != 'Cancelled'` | `status != 'Returned'` |

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {PROCUREMENT_METRICS}
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: >
    SELECT
      po.po_id,
      po.supplier_id,
      po.material_id,
      po.quantity,
      po.unit_cost,
      po.order_date,
      po.expected_delivery,
      po.actual_delivery,
      po.status,
      s.supplier_name,
      s.country,
      s.lead_time_days,
      s.reliability_score,
      s.category AS supplier_category,
      m.material_name,
      m.category AS material_category,
      m.unit_of_measure
    FROM {PURCHASE_ORDERS_TABLE} po
    JOIN {SUPPLIERS_TABLE} s ON po.supplier_id = s.supplier_id
    JOIN {MATERIALS_TABLE} m ON po.material_id = m.material_id
  filter: status != 'Cancelled'
  comment: Procurement KPIs joining purchase orders with supplier and material details. Excludes cancelled orders.
  dimensions:
    - name: supplier_name
      expr: supplier_name
      comment: Name of the supplier company
      synonyms:
        - supplier
        - vendor
    - name: country
      expr: country
      comment: Supplier country of origin
      synonyms:
        - supplier country
        - origin country
    - name: material_category
      expr: material_category
      comment: "Material type: Raw Material, Component, or Packaging"
      synonyms:
        - material type
        - category
    - name: material_name
      expr: material_name
      comment: Name of the material ordered
      synonyms:
        - material
        - item
    - name: supplier_category
      expr: supplier_category
      comment: "Supplier specialization: Raw Materials, Components, or Packaging"
    - name: order_month
      expr: DATE_TRUNC('MONTH', order_date)
      display_name: Order Month
      comment: Month the purchase order was placed
      synonyms:
        - month
        - order period
    - name: order_status
      expr: status
      comment: "PO status: Delivered, In Transit, or Pending"
  measures:
    - name: Total Spend
      expr: SUM(quantity * unit_cost)
      display_name: Total Spend
      comment: Total procurement spend (quantity x unit cost)
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - spend
        - procurement cost
        - total cost
    - name: Order Count
      expr: COUNT(DISTINCT po_id)
      display_name: Number of POs
      comment: Count of distinct purchase orders
      synonyms:
        - number of orders
        - PO count
    - name: Avg Lead Time
      expr: AVG(lead_time_days)
      display_name: Average Lead Time (days)
      comment: Average supplier lead time in days
      synonyms:
        - lead time
        - delivery time
    - name: Avg Unit Cost
      expr: AVG(unit_cost)
      display_name: Average Unit Cost
      comment: Average cost per unit across orders
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
    - name: On-Time Delivery Rate
      expr: |-
        ROUND(100.0 * COUNT(CASE WHEN actual_delivery <= expected_delivery THEN 1 END)
          / NULLIF(COUNT(actual_delivery), 0), 1)
      display_name: On-Time Delivery %
      comment: Percentage of delivered orders that arrived on or before expected date
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - on-time rate
        - OTD
        - delivery performance
$$
""")

print(f"✓ Created metric view: {PROCUREMENT_METRICS}")
print("  Dimensions: supplier_name, country, material_category, material_name, supplier_category, order_month, order_status")
print("  Measures: Total Spend, Order Count, Avg Lead Time, Avg Unit Cost, On-Time Delivery Rate")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {LOGISTICS_METRICS}
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: >
    SELECT
      sh.shipment_id,
      sh.origin_warehouse,
      sh.destination_city,
      sh.carrier_id,
      sh.ship_date,
      sh.estimated_delivery,
      sh.actual_delivery,
      sh.status,
      sh.weight_kg,
      sh.shipping_cost,
      c.carrier_name,
      c.transport_mode,
      c.service_region
    FROM {SHIPMENTS_TABLE} sh
    JOIN {CARRIERS_TABLE} c ON sh.carrier_id = c.carrier_id
  filter: status != 'Returned'
  comment: Logistics KPIs joining shipments with carrier details. Excludes returned shipments.
  dimensions:
    - name: carrier_name
      expr: carrier_name
      comment: Name of the shipping carrier
      synonyms:
        - carrier
        - shipper
    - name: transport_mode
      expr: transport_mode
      comment: "Mode of transport: Truck, Rail, Air, or Ocean"
      synonyms:
        - mode
        - shipping method
    - name: origin_warehouse
      expr: origin_warehouse
      comment: "Warehouse shipment originated from: WH-EAST, WH-WEST, or WH-CENTRAL"
      synonyms:
        - origin
        - warehouse
    - name: destination_city
      expr: destination_city
      comment: Delivery destination city
      synonyms:
        - destination
        - ship to
    - name: ship_month
      expr: DATE_TRUNC('MONTH', ship_date)
      display_name: Ship Month
      comment: Month the shipment departed
      synonyms:
        - month
        - shipping period
    - name: service_region
      expr: service_region
      comment: "Carrier coverage: Domestic, International, or Both"
    - name: shipment_status
      expr: status
      comment: "Shipment status: Delivered, In Transit, or Delayed"
  measures:
    - name: Total Shipping Cost
      expr: SUM(shipping_cost)
      display_name: Total Shipping Cost
      comment: Sum of all shipping costs in USD
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - shipping spend
        - freight cost
        - logistics cost
    - name: Shipment Count
      expr: COUNT(DISTINCT shipment_id)
      display_name: Number of Shipments
      comment: Count of distinct shipments
      synonyms:
        - number of shipments
        - shipment volume
    - name: Total Weight
      expr: SUM(weight_kg)
      display_name: Total Weight (kg)
      comment: Total shipped weight in kilograms
      synonyms:
        - weight
        - volume shipped
    - name: Avg Transit Days
      expr: |-
        AVG(DATEDIFF(actual_delivery, ship_date))
      display_name: Average Transit Days
      comment: Average days from ship date to actual delivery (delivered shipments only)
      synonyms:
        - transit time
        - delivery time
    - name: On-Time Rate
      expr: |-
        ROUND(100.0 * COUNT(CASE WHEN actual_delivery <= estimated_delivery THEN 1 END)
          / NULLIF(COUNT(actual_delivery), 0), 1)
      display_name: On-Time Delivery %
      comment: Percentage of shipments delivered on or before estimated date
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - on-time delivery
        - OTD
    - name: Avg Cost per KG
      expr: SUM(shipping_cost) / NULLIF(SUM(weight_kg), 0)
      display_name: Average Cost per KG
      comment: Average shipping cost per kilogram
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
$$
""")

print(f"✓ Created metric view: {LOGISTICS_METRICS}")
print("  Dimensions: carrier_name, transport_mode, origin_warehouse, destination_city, ship_month, service_region, shipment_status")
print("  Measures: Total Shipping Cost, Shipment Count, Total Weight, Avg Transit Days, On-Time Rate, Avg Cost per KG")

---
## Validation

In [0]:
print("=" * 60)
print("SUPPLY CHAIN DEMO - VALIDATION")
print("=" * 60)

all_tables = [
    (SUPPLIERS_TABLE, "Domain 1"),
    (MATERIALS_TABLE, "Domain 1"),
    (PURCHASE_ORDERS_TABLE, "Domain 1"),
    (INVENTORY_TABLE, "Domain 1"),
    (CARRIERS_TABLE, "Domain 2"),
    (SHIPMENTS_TABLE, "Domain 2"),
    (ROUTES_TABLE, "Domain 2"),
    (DELIVERY_EVENTS_TABLE, "Domain 2"),
]

print("\n┌──────────────────────────────────────────────────────────┐")
print("│ TABLE ROW COUNTS                                        │")
print("├──────────────────────────────────────────────────────────┤")
for tbl, domain in all_tables:
    count = spark.table(tbl).count()
    short_name = tbl.split(".")[-1]
    print(f"│  [{domain}] {short_name:<25} {count:>6} rows     │")

print("├──────────────────────────────────────────────────────────┤")
print("│ METRIC VIEWS                                            │")
print("├──────────────────────────────────────────────────────────┤")

# Test procurement metrics
try:
    pdf = spark.sql(f"SELECT `supplier_name`, MEASURE(`Total Spend`) AS spend FROM {PROCUREMENT_METRICS} GROUP BY ALL ORDER BY spend DESC LIMIT 3")
    print(f"│  ✓ procurement_metrics - OK                             │")
    print(f"│    Top spend: {pdf.collect()[0]['supplier_name']:<20} ${pdf.collect()[0]['spend']:>10,.2f}  │")
except Exception as e:
    print(f"│  ✗ procurement_metrics - ERROR: {str(e)[:30]}     │")

# Test logistics metrics
try:
    ldf = spark.sql(f"SELECT `carrier_name`, MEASURE(`Total Shipping Cost`) AS cost FROM {LOGISTICS_METRICS} GROUP BY ALL ORDER BY cost DESC LIMIT 3")
    print(f"│  ✓ logistics_metrics - OK                               │")
    print(f"│    Top cost: {ldf.collect()[0]['carrier_name']:<20} ${ldf.collect()[0]['cost']:>10,.2f}  │")
except Exception as e:
    print(f"│  ✗ logistics_metrics - ERROR: {str(e)[:30]}       │")

print("└──────────────────────────────────────────────────────────┘")
print("\n✓ All done! Ready for Genie space creation (notebook 02).")